# TikTok-10M dataset

This notebook analyses all ten official Parquet shards. It focuses on dataset integrity and the distribution of the per-video `comment_count` metadata field. The release does not contain comment text or individual comment rows.

Source: [official Hugging Face dataset](https://huggingface.co/datasets/The-data-company/TikTok-10M).

## 1. Verify the complete local download

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.parquet as pq
from IPython.display import display

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

ROOT = Path.cwd()
if not (ROOT / 'downloaded_datasets').exists() and (ROOT.parent / 'downloaded_datasets').exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'downloaded_datasets' / 'tiktok_10m'
PARQUET_FILES = sorted(DATA_DIR.glob('train-*-of-00010.parquet'))
if len(PARQUET_FILES) != 10:
    raise FileNotFoundError(f'Expected 10 Parquet shards, found {len(PARQUET_FILES)} in {DATA_DIR}')

inventory = []
for path in PARQUET_FILES:
    parquet = pq.ParquetFile(path)
    inventory.append({
        'file': path.name,
        'bytes': path.stat().st_size,
        'GiB': path.stat().st_size / 1024**3,
        'rows': parquet.metadata.num_rows,
        'columns': parquet.metadata.num_columns,
    })
inventory = pd.DataFrame(inventory)
display(inventory.round({'GiB': 3}))
print('Total rows:', f"{inventory['rows'].sum():,}")
print('Total local bytes:', f"{inventory['bytes'].sum():,}")
print('Total local GiB:', f"{inventory['bytes'].sum() / 1024**3:.2f}")

,file,bytes,GiB,rows,columns
0,train-00000-of-00010.parquet,962958914,0.90,1000000,55
1,train-00001-of-00010.parquet,985597059,0.92,1000000,55
2,train-00002-of-00010.parquet,990419541,0.92,1000000,55
3,train-00003-of-00010.parquet,989685117,0.92,1000000,55
4,train-00004-of-00010.parquet,985086257,0.92,1000000,55
5,train-00005-of-00010.parquet,979518366,0.91,1000000,55
6,train-00006-of-00010.parquet,973608580,0.91,1000000,55
7,train-00007-of-00010.parquet,971837702,0.91,1000000,55
8,train-00008-of-00010.parquet,968791860,0.90,1000000,55
9,train-00009-of-00010.parquet,971003940,0.90,1000000,55


Total rows: 10,000,000
Total local bytes: 9,778,507,336
Total local GiB: 9.11


## 2. Structure and released modalities

In [2]:
dataset = ds.dataset(DATA_DIR, format='parquet')
schema = dataset.schema
print(schema)

structure = pd.DataFrame([
    {'content': 'Video/post ID and URL', 'available': 'Yes', 'examples': 'id, url'},
    {'content': 'Creator identifier/profile metadata', 'available': 'Yes', 'examples': 'user_id, avatars, verified'},
    {'content': 'Description and hashtags', 'available': 'Yes', 'examples': 'desc, challenges'},
    {'content': 'Engagement counters', 'available': 'Yes', 'examples': 'comment, play, like, share, collect counts'},
    {'content': 'Music and point-of-interest metadata', 'available': 'Yes', 'examples': 'music_*, poi_*'},
    {'content': 'Comment text or comment rows', 'available': 'No', 'examples': 'comment_count only'},
    {'content': 'Embedded raw video/audio', 'available': 'No', 'examples': 'external URLs only'},
    {'content': 'Transcript, ASR, or OCR', 'available': 'No', 'examples': 'not included'},
])
display(structure)

id: int64
collected_time: string
create_time: int64
desc: string
duet_display: int64
duet_enabled: string
duet_info_duet_from_id: double
is_ad: string
item_comment_status: int64
item_mute: string
item_control_can_repost: string
official_item: string
original_item: string
share_enabled: string
playlist_id: double
stitch_display: int64
stitch_enabled: string
diversification_id: double
stats_time: string
collect_count: int64
comment_count: int64
digg_count: int64
play_count: int64
share_count: int64
vq_score: double
duration: int64
share_cover: string
poi_id: int64
poi_name: string
address: string
city: string
city_code: double
country_code: int64
poi_category: string
father_poi_id: string
father_poi_name: string
poi_tt_type_code: string
poi_tt_type_name_medium: string
poi_tt_type_name_super: string
poi_tt_type_name_tiny: string
user_id: int64
user_avatar_larger: string
user_avatar_medium: string
user_avatar_thumb: string
user_tt_seller: string
user_verified: string
challenges: string
mus

,content,available,examples
0,Video/post ID and URL,Yes,"id, url"
1,Creator identifier/profile metadata,Yes,"user_id, avatars, verified"
2,Description and hashtags,Yes,"desc, challenges"
3,Engagement counters,Yes,"comment, play, like, share, collect counts"
4,Music and point-of-interest metadata,Yes,"music_*, poi_*"
5,Comment text or comment rows,No,comment_count only
6,Embedded raw video/audio,No,external URLs only
7,"Transcript, ASR, or OCR",No,not included


## 3. Actual row, video, and creator counts

In [3]:
id_array = dataset.to_table(columns=['id']).column(0).combine_chunks()
user_array = dataset.to_table(columns=['user_id']).column(0).combine_chunks()

row_count = inventory['rows'].sum()
unique_video_ids = pc.count_distinct(id_array).as_py()
unique_user_ids = pc.count_distinct(user_array).as_py()

counts = pd.DataFrame([{
    'Parquet rows': row_count,
    'unique video IDs': unique_video_ids,
    'duplicate rows beyond first ID occurrence': row_count - unique_video_ids,
    'unique user IDs': unique_user_ids,
    'Hugging Face card claim': 10_000_000,
    'viewer estimate observed in Sept 2026': 6_654_004,
}])
display(counts)

print('The ten official files contain exactly 10 million rows. The viewer estimate is incomplete or stale.')

,Parquet rows,unique video IDs,duplicate rows beyond first ID occurrence,unique user IDs,Hugging Face card claim,viewer estimate observed in Sept 2026
0,10000000,9985545,14455,3165834,10000000,6654004


The ten official files contain exactly 10 million rows. The viewer estimate is incomplete or stale.


## 4. Duplicate video IDs

In [4]:
ids = id_array.to_numpy(zero_copy_only=False)
id_series = pd.Series(ids, name='id')
duplicate_ids = set(id_series[id_series.duplicated(keep=False)].tolist())

duplicate_rows = []
for path in PARQUET_FILES:
    part = pq.read_table(path, columns=['id', 'comment_count', 'play_count', 'digg_count', 'share_count', 'stats_time']).to_pandas()
    duplicate_rows.append(part[part['id'].isin(duplicate_ids)])
duplicate_rows = pd.concat(duplicate_rows, ignore_index=True)
duplicate_groups = duplicate_rows.groupby('id')

duplicate_summary = pd.DataFrame([{
    'IDs appearing more than once': len(duplicate_ids),
    'rows belonging to repeated IDs': len(duplicate_rows),
    'extra rows beyond one per ID': len(duplicate_rows) - len(duplicate_ids),
    'repeated IDs with different comment_count': int(duplicate_groups['comment_count'].nunique().gt(1).sum()),
    'repeated IDs with different stats_time': int(duplicate_groups['stats_time'].nunique().gt(1).sum()),
}])
display(duplicate_summary)
display(duplicate_groups.size().value_counts().sort_index().rename_axis('times_repeated').to_frame('video_IDs'))

,IDs appearing more than once,rows belonging to repeated IDs,extra rows beyond one per ID,repeated IDs with different comment_count,repeated IDs with different stats_time
0,9847,24302,14455,0,0


,video_IDs
times_repeated,
2,7739
3,590
4,1004
5,289
6,113
8,93
9,19


## 5. Comment statistics per unique video

In [5]:
id_comment = dataset.to_table(columns=['id', 'comment_count']).to_pandas()
unique_videos = id_comment.drop_duplicates('id')
comments = unique_videos['comment_count']

comment_stats = comments.describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999]).to_frame('comments_per_unique_video')
display(comment_stats)

additional = pd.DataFrame([{
    'unique videos': len(unique_videos),
    'total comment_count': int(comments.sum()),
    'mean comments per video': comments.mean(),
    'median comments per video': comments.median(),
    'videos with zero comments': int(comments.eq(0).sum()),
    'zero-comment percent': comments.eq(0).mean() * 100,
    'maximum comments': int(comments.max()),
}])
display(additional.round(2))

,comments_per_unique_video
count,"9,985,545.00"
mean,137.20
std,"1,420.63"
min,0.00
25%,3.00
50%,9.00
75%,34.00
90%,155.00
95%,410.00
99%,"2,301.00"


,unique videos,total comment_count,mean comments per video,median comments per video,videos with zero comments,zero-comment percent,maximum comments
0,9985545,1370040010,137.20,9.00,923332,9.25,1070474


## 6. Interesting point: comments are strongly concentrated

In [6]:
sorted_comments = comments.sort_values(ascending=False)
total_comments = sorted_comments.sum()
concentration = []
for fraction in [0.001, 0.01, 0.05, 0.10]:
    top_n = int(np.ceil(len(sorted_comments) * fraction))
    concentration.append({
        'top_video_fraction': f'{fraction * 100:g}%',
        'number_of_videos': top_n,
        'share_of_all_comment_count': sorted_comments.iloc[:top_n].sum() / total_comments * 100,
    })
display(pd.DataFrame(concentration).round(2))

,top_video_fraction,number_of_videos,share_of_all_comment_count
0,0.1%,9986,20.72
1,1%,99856,51.71
2,5%,499278,78.54
3,10%,998555,87.72


## 7. Comment counts by advertisement and verified status

In [7]:
group_results = []
for field in ['is_ad', 'user_verified']:
    for value in ['f', 't']:
        values = dataset.to_table(
            columns=['comment_count'],
            filter=ds.field(field) == value,
        ).column(0).combine_chunks().to_numpy(zero_copy_only=False)
        group_results.append({
            'field': field,
            'value': value,
            'rows': len(values),
            'mean_comments': values.mean(),
            'median_comments': np.median(values),
            'zero_comment_percent': (values == 0).mean() * 100,
            'maximum_comments': values.max(),
        })
display(pd.DataFrame(group_results).round(2))

print('These are descriptive associations. They do not show that ad or verification status causes engagement.')

,field,value,rows,mean_comments,median_comments,zero_comment_percent,maximum_comments
0,is_ad,f,9930638,137.34,9.00,9.26,1070474
1,is_ad,t,69362,110.67,12.00,7.17,604146
2,user_verified,f,9851267,132.04,9.00,9.35,1070474
3,user_verified,t,148733,475.65,48.00,2.34,604146


These are descriptive associations. They do not show that ad or verification status causes engagement.


## 8. Relationship between comments and other engagement counters

In [8]:
# Use a deterministic 10% sample to keep the rank-correlation calculation memory-efficient.
rng = np.random.default_rng(20260922)
sample_indices = np.sort(rng.choice(row_count, size=1_000_000, replace=False))
engagement_columns = ['comment_count', 'play_count', 'digg_count', 'share_count', 'collect_count']
sample = {}
for column in engagement_columns:
    values = dataset.to_table(columns=[column]).column(0).combine_chunks().to_numpy(zero_copy_only=False)
    sample[column] = values[sample_indices]
sample = pd.DataFrame(sample)

display(sample.corr(method='spearman').round(4))
print('Spearman correlations use a fixed one-million-row sample. They describe association, not causality.')

,comment_count,play_count,digg_count,share_count,collect_count
comment_count,1.00,0.65,0.73,0.66,0.68
play_count,0.65,1.00,0.89,0.80,0.85
digg_count,0.73,0.89,1.00,0.79,0.86
share_count,0.66,0.80,0.79,1.00,0.85
collect_count,0.68,0.85,0.86,0.85,1.00


Spearman correlations use a fixed one-million-row sample. They describe association, not causality.


## 9. Publication and collection windows

In [9]:
windows = {'create_time': [], 'collected_time': [], 'stats_time': []}
for path in PARQUET_FILES:
    part = pq.read_table(path, columns=list(windows)).to_pandas()
    windows['create_time'].append(pd.to_datetime(part['create_time'], unit='s', utc=True, errors='coerce'))
    windows['collected_time'].append(pd.to_datetime(part['collected_time'], utc=True, errors='coerce'))
    windows['stats_time'].append(pd.to_datetime(part['stats_time'], utc=True, errors='coerce'))

time_summary = []
for field, parts in windows.items():
    time_summary.append({
        'field': field,
        'minimum': min(part.min() for part in parts),
        'maximum': max(part.max() for part in parts),
    })
display(pd.DataFrame(time_summary))

,field,minimum,maximum
0,create_time,2021-09-16 03:36:56+00:00,2025-07-02 15:20:32+00:00
1,collected_time,2025-03-07 17:08:54.866064+00:00,2025-07-03 03:11:59.640006+00:00
2,stats_time,2025-03-07 17:48:55.322076+00:00,2025-07-03 03:27:47.634966+00:00


# Findings

- All ten official Parquet files are locally available and contain exactly **10,000,000 rows**. The Hugging Face viewer's roughly 6.65-million estimate does not describe the complete downloaded files.
- The table contains **9,985,545 unique video IDs** and **3,165,834 user IDs**. There are 14,455 extra repeated rows across 9,847 repeated video IDs. The checked engagement fields and statistics timestamps are identical within those repeats.
- After keeping one row per video ID, the mean is about **137.20 comments per video**, while the median is **9**. The large difference shows a strongly right-skewed distribution.
- The top 1% of unique videos account for about **51.71%** of all recorded `comment_count`. About 9.25% of unique videos have zero comments.
- Verified accounts have much higher descriptive comment counts than unverified accounts in this corpus, but this is not a causal result and the dataset is biased toward trending US-point-of-interest content.
- `comment_count` is a metadata snapshot. The dataset contains no comment text, author IDs for commenters, comment timestamps, or reply structure.